## задача
1. проанализировать эти данные секвенирования, чтобы определить мутации, ответственные за устойчивость E. coli к антибиотикам
2. изучить гены, которые мутировали, чтобы определить механизм устойчивости к антибиотикам в каждом случае
3. дать рекомендации по альтернативным антибиотикам, которые врач мог бы использовать для лечения каждого штамма

### подсказки
1 - forward, 2 - rewerse

In [ ]:
#1 - скачивание сырых данных и проверка их целостности
wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/005/845/GCF_000005845.2_ASM584v2/GCF_000005845.2_ASM584v2_genomic.fna.gz #fasta последовательность генома немутировавшей E. coli K-12 MG1655
wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/005/845/GCF_000005845.2_ASM584v2/GCF_000005845.2_ASM584v2_genomic.gff.gz #аннотация её генома
# информация об этом геноме здесь https://www.ncbi.1.1K nlm.nih.gov/datasets/genome/GCF_000005845.2/
#загрузка через браузер необработанных данные секвенирования штамма E. coli, устойчивого к антибиотику ампициллину. Названия файлов - amp_res_1.fastq и amp_res_2.fastq
#вручную извлекла файлы из архивов
#с помощью команд проверила целостность файлов
head -20 amp_res_1.fastq #вывести первые 20 строк
wc -l amp_res_1.fastq #посмотреть количество прочтений. В обоих файлах одинако - 1823504 строчки, значит, 1823504/4=455876 прочтений
#устновка ПО, просмотр статистик по fastq файлу
conda install -c bioconda seqkit
seqkit stats amp_res_1.fastq

In [ ]:
#статистики 
#forward
file             format  type  num_seqs     sum_len  min_len  avg_len  max_len
amp_res_1.fastq  FASTQ   DNA    455,876  46,043,476      101      101      101
#reverse
ats amp_res_2.fastq
file             format  type  num_seqs     sum_len  min_len  avg_len  max_len
amp_res_2.fastq  FASTQ   DNA    455,876  46,043,476      101      101      101

In [ ]:
#работа в fastqc - проверка данных секвенирования
conda activate fastqc
fastqc -o . /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/raw_data/amp_res_1.fastq /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/raw_data/amp_res_2.fastq
#качество хорошее - есть немного адаптеров и снижение качества секвенирования к концу прочтения, что типично для Illumina. Количество прочтений совпадает - также 455876
#Per tile sequence quality в forward - проблема, может указывать на техническую ошибку секвенирования

In [ ]:
#фильтрация ридов trimmomatic
conda install -c bioconda trimmomatic 

trimmomatic PE -phred33 \ #запуск
/mnt/d/data/analysis_of_genomic_and_transcriptomic_data/raw_data/amp_res_1.fastq \
/mnt/d/data/analysis_of_genomic_and_transcriptomic_data/raw_data/amp_res_2.fastq \
amp_res_1_paired.fastq amp_res_1_unpaired.fastq \
amp_res_2_paired.fastq amp_res_2_unpaired.fastq \
LEADING:20 TRAILING:20 SLIDINGWINDOW:10:20 MINLEN:20 
#Обрезать основания в начале рида, если качество ниже 20.
#Обрезать основания в конце рида, если качество ниже 20.
#Обрезать риды с использованием подхода скользящего окна с размером окна 10 и средним качеством внутри окна 20.
#Отбросить рид, если его длина меньше 20.
#количество прочтений после фильтрации - 446,259 
#запуск fastqc
fastqc -o . /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/processed_data/trimmomatic/amp_res_1_paired.fastq /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/processed_data/trimmomatic/amp_res_2_paired.fastq
#лучше Per base sequence quality, но хуже Sequence Length Distribution. Изменилась длина сиквенсов и немного уменьшился размер файла

In [ ]:
#выравнивание по алгоритму BWA-MEM
conda install bwa
bwa index GCF_000005845.2_ASM584v2_genomic.fna #индексирование референса
#выходные файлы в папке raw_data
#выравнивание
bwa mem GCF_000005845.2_ASM584v2_genomic.fna amp_res_1.fastq amp_res_2.fastq > alignment.sam
#сжатие SAM-файла в BAM-файл
samtools view -S -b alignment.sam > alignment.bam
#получение статистики

статистика
99.87% прочтений картировано

912095 + 0 in total (QC-passed reads + QC-failed reads)
911752 + 0 primary
0 + 0 secondary
343 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
910940 + 0 mapped (99.87% : N/A)
910597 + 0 primary mapped (99.87% : N/A)
911752 + 0 paired in sequencing
455876 + 0 read1
455876 + 0 read2
907476 + 0 properly paired (99.53% : N/A)
909488 + 0 with itself and mate mapped
1109 + 0 singletons (0.12% : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)

In [ ]:
#создаём pileup-файл - таблица, в которой видно для каждой позиции генома покрытие, основание, качество основания и SNP, если есть. Отличить мутацию от ошибки секвенирования
samtools mpileup -f GCF_000005845.2_ASM584v2_genomic.fna alignment_sorted.bam > my.mpileup #на выходе файл my.mpileu
#ищем SNP с порогом 0.05 (минимальный процент нереференсных оснований в позиции, необходимый для определения мутации в образце)
varscan mpileup2snp my.mpileup --min-var-freq 0.05 --variants --output-vcf 1 > VarScan_results.vcf

In [ ]:
#автоматическая аннотация SNP - SnpEff 
#создадние текстового файла snpEff.config, в нём фраза k12.genome : ecoli_K12
#создание папки для базы данных mkdir -p data/k12, туда же файл VarScan_results.vcf
#скачивание .gb файла, его разархивирование и переименовывание
gunzip GCF_000005845.2_ASM584v2_genomic.gbff.gz
cp GCF_000005845.2_ASM584v2_genomic.gbff genes.gbk

#создание базы данных
/usr/lib/jvm/java-21-openjdk-amd64/bin/java -jar /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/raw_data/data/k12/snpEff/snpEff.jar build -genbank -v k12
#аннотация
/usr/lib/jvm/java-21-openjdk-amd64/bin/java -jar /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/raw_data/data/k12/snpEff/snpEff.jar ann k12 VarScan_results.vcf > VarScan_results_annotated.vcf